# ⚡ Notebook 4: Energy Access & Development in Ghana

> **Objective:** Energy access is a development multiplier — what does Ghana's electrification data reveal about where progress is happening and where it isn't?

This notebook connects to my MSc dissertation on energy poverty detection using smart meter data — looking at the policy and access side of the same problem from a developing economy perspective.

**Data source:** World Bank Open Data (`data/raw/energy_indicators.csv`)

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, warnings
sys.path.append('../src')
from plot_utils import apply_style, save_fig, format_pct, PALETTE

warnings.filterwarnings('ignore')
apply_style()

energy  = pd.read_csv('../data/raw/energy_indicators.csv')
poverty = pd.read_csv('../data/raw/poverty_indicators.csv')

gh_energy  = energy[energy['country']  == 'Ghana'].sort_values('date').reset_index(drop=True)
gh_poverty = poverty[poverty['country'] == 'Ghana'].sort_values('date').reset_index(drop=True)
print(f'Loaded energy: {len(gh_energy)} | poverty: {len(gh_poverty)} Ghana records')

## 2. Chart: National Electrification — Urban vs Rural Gap

In [ ]:
elec = gh_energy.dropna(subset=['electricity_access_pct', 'electricity_rural_pct', 'electricity_urban_pct'])

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(elec['date'], elec['electricity_urban_pct'], color=PALETTE[0],
        linewidth=2.5, label='Urban electrification', marker='o', markersize=4)
ax.plot(elec['date'], elec['electricity_rural_pct'], color=PALETTE[1],
        linewidth=2.5, label='Rural electrification', marker='s', markersize=4)
ax.plot(elec['date'], elec['electricity_access_pct'], color='#374151',
        linewidth=1.8, label='National average', linestyle='--')

ax.fill_between(elec['date'], elec['electricity_rural_pct'], elec['electricity_urban_pct'],
                alpha=0.08, color=PALETTE[1], label='Urban-rural gap')

# Annotate latest gap
latest = elec.iloc[-1]
gap = latest['electricity_urban_pct'] - latest['electricity_rural_pct']
ax.text(latest['date'] - 1, (latest['electricity_urban_pct'] + latest['electricity_rural_pct'])/2,
        f'{gap:.0f}pp gap', fontsize=10, color=PALETTE[1], fontweight='bold')

format_pct(ax)
ax.set_title('Ghana Electricity Access: Urban vs Rural (2000–2023)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Population with Electricity Access (%)')
ax.set_ylim(0, 105)
ax.legend()
sns.despine()
plt.tight_layout()
save_fig(fig, '04_electricity_urban_rural.png')
plt.show()

## 3. Chart: Energy Access vs Human Development Correlation

In [ ]:
# Merge energy + poverty data on year
merged = pd.merge(
    gh_energy[['date', 'electricity_access_pct']],
    gh_poverty[['date', 'primary_school_enrolment', 'infant_mortality_per_1000']],
    on='date', how='inner'
).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Energy access vs school enrolment
axes[0].scatter(merged['electricity_access_pct'], merged['primary_school_enrolment'],
                color=PALETTE[0], s=60, alpha=0.7)
# Trend line
z = np.polyfit(merged['electricity_access_pct'].dropna(),
               merged['primary_school_enrolment'].dropna(), 1)
p = np.poly1d(z)
x_range = np.linspace(merged['electricity_access_pct'].min(), merged['electricity_access_pct'].max(), 100)
axes[0].plot(x_range, p(x_range), color=PALETTE[1], linestyle='--', linewidth=1.5)
axes[0].set_title('Electricity Access vs\nPrimary School Enrolment', fontweight='bold')
axes[0].set_xlabel('Electricity Access (%)')
axes[0].set_ylabel('Primary Enrolment (%)')
sns.despine(ax=axes[0])

# Energy access vs infant mortality (inverse relationship)
axes[1].scatter(merged['electricity_access_pct'], merged['infant_mortality_per_1000'],
                color=PALETTE[1], s=60, alpha=0.7)
z2 = np.polyfit(merged['electricity_access_pct'].dropna(),
                merged['infant_mortality_per_1000'].dropna(), 1)
p2 = np.poly1d(z2)
axes[1].plot(x_range, p2(x_range), color=PALETTE[0], linestyle='--', linewidth=1.5)
axes[1].set_title('Electricity Access vs\nInfant Mortality Rate', fontweight='bold')
axes[1].set_xlabel('Electricity Access (%)')
axes[1].set_ylabel('Infant Deaths per 1,000 Live Births')
sns.despine(ax=axes[1])

plt.suptitle('Energy Access as a Development Multiplier: Ghana', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, '04_energy_development_correlation.png')
plt.show()

corr_school   = merged['electricity_access_pct'].corr(merged['primary_school_enrolment'])
corr_mortality = merged['electricity_access_pct'].corr(merged['infant_mortality_per_1000'])
print(f'Correlation — electricity vs school enrolment: {corr_school:.3f}')
print(f'Correlation — electricity vs infant mortality:  {corr_mortality:.3f}')

## 4. Chart: Renewable Energy Share

In [ ]:
renew = gh_energy.dropna(subset=['renewable_electricity_pct'])

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(renew['date'], renew['renewable_electricity_pct'],
                color=PALETTE[0], alpha=0.3)
ax.plot(renew['date'], renew['renewable_electricity_pct'],
        color=PALETTE[0], linewidth=2.5)
ax.set_title('Renewable Energy as % of Electricity Output — Ghana', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Renewable Electricity (%)')
format_pct(ax)
sns.despine()
plt.tight_layout()
save_fig(fig, '04_renewable_energy.png')
plt.show()

## 5. Final Summary: The Ghana Development Story in Data

### 📌 Across all four notebooks, four themes emerge:

**1. Growth without guaranteed inclusion**
Ghana has grown impressively — but the Gini coefficient tells the story growth alone cannot: rising incomes have not automatically closed regional, gender, or urban-rural gaps.

**2. Mobile technology as the leapfrog mechanism**
Where traditional banking failed to reach, mobile money succeeded. Ghana's fintech trajectory is not about replacing banks — it is about reaching the 40%+ who traditional banks never served.

**3. Energy access is not optional infrastructure — it's a development multiplier**
The correlation between electrification rates and both school enrolment and infant mortality is clear in the data. Energy poverty is not just an energy problem — it is a health, education, and economic problem.

**4. Fragility beneath the headline numbers**
The 2022 debt crisis erased fiscal gains built over a decade. Ghana's development story is real — but it is built on a narrower foundation than GDP per capita suggests.

---

### 💼 Why This Matters Beyond Ghana

| Insight | Broader Application |
|---------|--------------------|
| Mobile money outpacing banking | Blueprint for financial inclusion in underserved markets globally |
| Energy-health-education link | Argues for integrated development policy, not siloed sector funding |
| Gender gaps in formal finance | Fintech product design opportunity — not just a policy problem |
| Debt fragility | Commodity-dependent economies need diversification for sustainable growth |

---

### 🔗 Personal Note

I built this analysis because numbers about Ghana are rarely told in data — they are told in headlines. My MSc dissertation detected energy poverty in UK smart meter data at 99.6% recall; this project sits on the other side of the same problem — what does energy poverty look like from a policy and access perspective in a country still building its grid? The data here is a starting point, not a conclusion.